**Submitted by:** Achira Sadharanga  
**Student ID:** CIT-23-02-0170  
**Contribution:** Task 1 and Task 2


# SmartCare Hospital AI Coursework — Patient Readmission Prediction

**Module:** CCS3440 — Artificial Intelligence

**Selected Option:** Option B — Patient Readmission Prediction (30-day readmission)

**Target Variable:** `readmitted_30_days`

**Problem Type:** Binary Classification

This notebook implements Tasks 2-8 of the coursework specification using the SmartCare
Hospital AI dataset (`smartcare_ai_dataset_1000.csv`, 1000 records, 33 attributes).

**Notebook structure**
0. Environment Setup (Jupyter)
1. Setup & Imports
2. Task 2 — Dataset Understanding
3. Task 3 — Data Preprocessing & Feature Engineering
4. Task 4 — Exploratory Data Analysis
5. Task 5 — Machine Learning Model Development
6. Task 6 — Model Evaluation
7. Task 7 — Explainable AI Analysis (SHAP)
8. Task 8 — Model Export for Prototype



## 0. Environment Setup (Jupyter)

This notebook is designed to run in a local Jupyter environment. The cell below:

1. Installs the two packages that may not be pre-installed (`xgboost`, `shap`).
2. Creates the local `data/`, `figs/` and `models/` folders used throughout the notebook.
3. Checks that the two dataset files are present in the `data/` folder:
   `smartcare_ai_dataset_1000.csv` and `smartcare_ai_dataset_data_dictionary.csv`.

**Before running:** place both CSV files in a `data/` folder next to this notebook.

Run this cell first, every time, before anything else.

In [2]:
import os, sys, subprocess

def _pip_install_quiet(*packages):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

try:
    import xgboost, shap
except ImportError:
    _pip_install_quiet('xgboost', 'shap')

os.makedirs('data', exist_ok=True)
os.makedirs('figs', exist_ok=True)
os.makedirs('models', exist_ok=True)

DATA_FILE = 'smartcare_ai_dataset_1000.csv'
DICT_FILE = 'smartcare_ai_dataset_data_dictionary.csv'

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f"Could not find {DATA_FILE}. "
        "Make sure the dataset is in the same folder as the notebook."
    )
print("Environment ready.")
print("Data file present:", os.path.exists(DATA_FILE))
print("Data dictionary present:", os.path.exists(DICT_FILE))

Environment ready.
Data file present: True
Data dictionary present: True


## 1. Setup and Imports

In [40]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                              RocCurveDisplay, classification_report)

import shap
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titleweight'] = 'bold'

pd.set_option('display.max_columns', 40)


## 2. Task 2 — Dataset Understanding

### 2.1 Loading the data

In [43]:
df = pd.read_csv(DATA_FILE)
data_dict = pd.read_csv(DICT_FILE)

print(f"Records: {df.shape[0]}, Attributes: {df.shape[1]}")
df.head()

Records: 1000, Attributes: 33


,record_id,patient_id,age,gender,blood_group,department,diagnosis,appointment_date,waiting_days,previous_appointments,missed_previous_appointments,appointment_status,admitted,room_type,length_of_stay_days,previous_admissions,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,lab_tests_count,treatments_count,consultation_fee_lkr,room_charge_lkr,lab_charge_lkr,medicine_charge_lkr,total_bill_lkr,payment_status,payment_method,no_show,readmitted_30_days,disease_risk_level
0,1,P10001,53,Male,A-,General Medicine,Migraine,2025-04-10,10,1,0,Completed,0,NaN,0,1,127,75,117,211,26.1,0,3,2000,0,0,11596,13596,Paid,Insurance,0,0,High
1,2,P10002,26,Male,B-,General Medicine,Diabetes,2025-05-15,2,3,1,Completed,0,NaN,0,0,130,73,136,173,32.8,0,1,2000,0,0,3652,5652,Paid,Insurance,0,0,Medium
2,3,P10003,22,Male,B+,Orthopedics,Back Pain,2025-07-09,22,7,1,No-Show,0,NaN,0,1,141,64,90,176,29.4,1,0,2500,0,1200,2562,6262,Unpaid,Insurance,1,0,Medium
3,4,P10004,44,Female,AB-,Cardiology,Asthma,2025-10-16,16,1,0,Completed,0,NaN,0,0,124,82,126,189,24.9,2,1,2000,0,5000,10262,17262,Paid,Online,0,0,Medium
4,5,P10005,51,Female,O+,Neurology,Hypertension,2025-12-18,12,4,0,Scheduled,0,NaN,0,1,119,81,65,195,27.0,2,0,4000,0,6000,10414,20414,Paid,Cash,0,0,Medium


### 2.2 Data dictionary

In [42]:
data_dict


,Column,Description
0,record_id,Unique row identifier
1,patient_id,Synthetic patient identifier
2,age,Patient age in years
3,gender,Patient gender
4,blood_group,Patient blood group
5,department,Hospital department
6,diagnosis,Primary diagnosis category
7,appointment_date,Appointment date
8,waiting_days,Number of days between booking and appointment
9,previous_appointments,Number of previous appointments


### 2.3 Structural overview

`df.info()` confirms data types for every attribute and shows there are no `null`
entries at the raw-file level (the apparent "missing" values in `room_type` are in
fact a structural NaN that only appears for patients who were never admitted — this
is examined in Section 3).

In [44]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 33 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   record_id                     1000 non-null   int64  
 1   patient_id                    1000 non-null   object 
 2   age                           1000 non-null   int64  
 3   gender                        1000 non-null   object 
 4   blood_group                   1000 non-null   object 
 5   department                    1000 non-null   object 
 6   diagnosis                     1000 non-null   object 
 7   appointment_date              1000 non-null   object 
 8   waiting_days                  1000 non-null   int64  
 9   previous_appointments         1000 non-null   int64  
 10  missed_previous_appointments  1000 non-null   int64  
 11  appointment_status            1000 non-null   object 
 12  admitted                      1000 non-null   int64  
 13  room

In [7]:
df.describe(include='number').T


,count,mean,std,min,25%,50%,75%,max
record_id,1000.0,500.5000,288.819436,1.0,250.75,500.5,750.25,1000.0
age,1000.0,44.7440,17.851474,1.0,33.00,44.0,57.00,90.0
waiting_days,1000.0,21.8510,13.040892,0.0,11.00,22.0,34.00,44.0
previous_appointments,1000.0,2.8840,1.693343,0.0,2.00,3.0,4.00,10.0
missed_previous_appointments,1000.0,0.5530,0.740094,0.0,0.00,0.0,1.00,4.0
admitted,1000.0,0.3300,0.470448,0.0,0.00,0.0,1.00,1.0
length_of_stay_days,1000.0,1.1030,1.885191,0.0,0.00,0.0,2.00,9.0
previous_admissions,1000.0,0.8550,0.958590,0.0,0.00,1.0,1.00,5.0
systolic_bp,1000.0,128.4230,15.486895,85.0,117.00,128.0,139.00,178.0
diastolic_bp,1000.0,78.8280,10.043669,50.0,72.00,79.0,86.00,111.0


### 2.4 Target variable and population definition

`readmitted_30_days` records whether a patient was re-admitted to hospital within 30
days of a prior admission. By definition, a patient who was **never admitted in the
first place cannot be "re-admitted"**. The crosstab below confirms this: every record
with `admitted = 0` has `readmitted_30_days = 0`, while the target only takes a
meaningful value of 1 for previously admitted patients.

In [8]:
pd.crosstab(df['admitted'], df['readmitted_30_days'], margins=True)


readmitted_30_days,0,1,All
admitted,,,
0,670,0,670
1,77,253,330
All,747,253,1000


This has a direct implication for problem framing (discussed further in Task 3):
if the model were trained on the *whole* dataset (including outpatients who were
never admitted), `admitted` would act as a near-perfect leaking predictor and the
resulting model would trivially learn "not admitted -> not readmitted" rather than
learning the clinically meaningful patterns that separate readmitted patients from
patients who were admitted once and did **not** come back. The coursework population
of interest is therefore restricted to previously **admitted** patients
(`admitted == 1`), which is standard practice in the clinical readmission-prediction
literature (e.g. Halac et al., 2025, who restrict their 30-day readmission model to
adult admissions rather than the full outpatient population).

In [9]:
admitted_share = df['admitted'].mean()
readmit_share_overall = df['readmitted_30_days'].mean()
readmit_share_admitted = df.loc[df['admitted']==1, 'readmitted_30_days'].mean()
print(f"Share of records admitted: {admitted_share:.1%}")
print(f"Readmission rate over ALL records: {readmit_share_overall:.1%}")
print(f"Readmission rate among ADMITTED patients: {readmit_share_admitted:.1%}")


Share of records admitted: 33.0%
Readmission rate over ALL records: 25.3%
Readmission rate among ADMITTED patients: 76.7%


### 2.5 Attribute grouping

Using the data dictionary, the 33 attributes fall into five functional groups:

| Group | Attributes |
|---|---|
| Identifiers | `record_id`, `patient_id` |
| Demographics | `age`, `gender`, `blood_group` |
| Clinical | `diagnosis`, `systolic_bp`, `diastolic_bp`, `blood_sugar_mg_dl`, `cholesterol_mg_dl`, `bmi` |
| Operational | `department`, `appointment_date`, `waiting_days`, `previous_appointments`, `missed_previous_appointments`, `appointment_status`, `admitted`, `room_type`, `length_of_stay_days`, `previous_admissions`, `lab_tests_count`, `treatments_count` |
| Financial | `consultation_fee_lkr`, `room_charge_lkr`, `lab_charge_lkr`, `medicine_charge_lkr`, `total_bill_lkr`, `payment_status`, `payment_method` |
| AI targets | `no_show`, `readmitted_30_days`, `disease_risk_level` |

Because `no_show` and `disease_risk_level` are targets for the *other* two coursework
options, they are excluded from the input feature set used in Task 5 to avoid
leaking outcome information from parallel tasks into the readmission model.